# phase2_pulsedb — retrain DiBP-Net on PulseDB (real subject IDs)

## Why
UCI ships no patient IDs, so `phase1.ipynb::extract_uci` keys "subject" on the record index; its splits are record-independent, not subject-independent (see paper Section 5.1). PulseDB has real subject IDs (**2,845 subjects, 15.8M beats**), so training here gives a genuinely subject-independent in-domain number, and the gap from UCI's 4.18 mmHg is the point.

## Design (why the notebook is built this way)
PulseDB is far larger than UCI, so the notebook streams data lazily and reuses phase2's model instead of loading everything up front. Three choices:
1. **Subsample subject-balanced, session-spread** — `SEG_PER_SUBJ` segments evenly spaced across each subject's recording, not the first N beats (which would bias toward the session start where the label drifts; PulseDB SBP ICC = 0.425).
2. **Never materialise windows** — cut on the fly from a memmap in `__getitem__`.
3. **Reuse phase2's model/trainer verbatim** — exec phase2's cells and swap only `_make_tensor_dataset`, so `DiBPNet` stays bit-identical to the paper's.

## Stages (each idempotent — safe to re-run)
A select beats → compact memmap · B subject split + window centres · C lazy `SPLITS` · D train · E evaluate vs UCI.


In [ ]:
# BOOTSTRAP — pull in phase2's definitions (imports, config, DiBPNet, losses,
# trainer) by exec-ing its definition cells only, so no experiment auto-runs and
# there is exactly one DiBPNet in the project.

import json, os, sys, time, gc

PHASE2_NB  = "phase2.ipynb"
CORE_CELLS = range(0, 20)          # definitions only

def load_phase2_core(nb_path=PHASE2_NB, cells=CORE_CELLS):
    """Exec phase2's definition cells. ABORTS on the first failure.

    Fail-fast is deliberate. These cells form an import chain: cell 2 imports
    torch, cell 9 defines DiBPNet with it, cell 15 trains it. If cell 2 dies (a
    missing package, say) and we carry on, cells 3/7/9/15/17/19 all die with
    NameError and the one real cause ends up buried under six fake ones.
    """
    nb = json.load(open(nb_path))
    ran = []
    for i in cells:
        if i >= len(nb["cells"]):
            break
        c = nb["cells"][i]
        if c["cell_type"] != "code":
            continue
        src = "".join(c["source"])
        if not src.strip():
            continue
        try:
            exec(compile(src, f"<{nb_path}:cell{i}>", "exec"), globals())
        except Exception as e:
            print(f"  exec'd OK up to: {ran}")
            raise RuntimeError(
                f"phase2.ipynb cell {i} failed: {type(e).__name__}: {e}\n"
                f"  Every later cell depends on it. Fix the cause -- do not skip the\n"
                f"  cell. If a package is missing, install it in THIS kernel."
            ) from e
        ran.append(i)
    print(f"  exec'd phase2 cells: {ran}")
    return True

load_phase2_core()

for name in ("DiBPNet", "_run_batch", "_run_epoch", "_two_phase_train",
             "_make_optimizer", "quick_metrics", "_make_tensor_dataset",
             "_empty_hist", "save_hist_csv", "DataSplits", "Split",
             "ARCH", "TRAIN", "CALIB", "DEVICE", "HTN_SBP", "HTN_DBP"):
    assert name in globals(), f"missing from phase2 core: {name}"
print(f"\n  \u2705 phase2 core loaded.  DEVICE={DEVICE}  ARCH.batch={ARCH.batch}  "
      f"embed_size={ARCH.embed_size}")
print(f"  epochs: p1={TRAIN.epochs_p1} + p2={TRAIN.epochs_p2}")

In [ ]:
# CONFIG — PulseDB output paths, redirected so this notebook can never overwrite
# the UCI checkpoints the paper reports.

import numpy as np, pandas as pd, torch
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import Dataset, DataLoader

PDB_RAW    = "outputs/phase_1/data/pulsedb_uncompressed"   # the .npy files (mmap-able)
PDB_SUBSET = "outputs/phase_1/data/pulsedb_subset"          # stage A writes here

OUT_DIR    = "outputs/phase_2_pulsedb"
CKPT_DIR   = os.path.join(OUT_DIR, "checkpoints")
EXPORT_DIR = os.path.join(OUT_DIR, "exports")
FIG_DIR    = os.path.join(OUT_DIR, "figures")
HIST_DIR   = os.path.join(OUT_DIR, "hist")
for _d in (PDB_SUBSET, OUT_DIR, CKPT_DIR, EXPORT_DIR, FIG_DIR, HIST_DIR):
    os.makedirs(_d, exist_ok=True)

# ── subsampling budget ───────────────────────────────────────────────────────
#  SEG_PER_SUBJ contiguous segments of SEG_LEN beats, spread evenly across each
#  subject's recording. Windows are 11 beats, so SEG_LEN must be >= 11; at
#  stride 3 a 25-beat segment yields 5 windows.
SEG_PER_SUBJ = 12
SEG_LEN      = 25                      # -> <= 300 beats/subject
SPLIT_SEED   = 42
TEST_FRAC    = 0.15
VAL_FRAC     = 0.15

HALF   = ARCH.n_beats // 2             # 5  -> 11-beat window
STRIDE = ARCH.window_stride            # 3
assert SEG_LEN >= 2 * HALF + 1, "segment shorter than one window"

_sid = np.load(f"{PDB_RAW}/subject_ids.npy", mmap_mode="r")
_n_beats_total = len(_sid)
_n_subj_total  = len(np.unique(np.asarray(_sid)))
print(f"  PulseDB raw : {_n_beats_total:,} beats   {_n_subj_total:,} REAL subjects")
print(f"  budget      : {SEG_PER_SUBJ} x {SEG_LEN} = {SEG_PER_SUBJ*SEG_LEN} beats/subject max")
print(f"  -> at most  : {_n_subj_total*SEG_PER_SUBJ*SEG_LEN:,} beats  "
      f"(~{_n_subj_total*SEG_PER_SUBJ*(SEG_LEN-2*HALF)//STRIDE:,} windows)")
assert _n_subj_total <= ARCH.embed_size, "subject embedding too small"
del _sid

In [ ]:
# STAGE A — select beats into a compact memmap.
# Take SEG_PER_SUBJ contiguous segments of SEG_LEN beats, evenly spaced across each
# subject's recording (not the first N beats: BP drifts within a session, PulseDB
# SBP ICC = 0.425, so the first beats are a biased slice). Idempotent.

_MANIFEST = os.path.join(PDB_SUBSET, "manifest.json")
_WANT = dict(seg_per_subj=SEG_PER_SUBJ, seg_len=SEG_LEN, source=PDB_RAW)

def _subset_is_current():
    if not os.path.exists(_MANIFEST):
        return False
    try:
        got = json.load(open(_MANIFEST))
    except Exception:
        return False
    if {k: got.get(k) for k in _WANT} != _WANT:
        print("  manifest differs from the current config — rebuilding")
        return False
    need = ["X_ppg.npy", "Y_abp.npy", "Y_bp.npy", "dur.npy", "subj.npy", "seg.npy"]
    return all(os.path.exists(os.path.join(PDB_SUBSET, f)) for f in need)


def build_subset():
    if _subset_is_current():
        print(f"  ✅ subset already built ({json.load(open(_MANIFEST))['n_beats']:,} beats) — skipping")
        return

    t0 = time.time()
    sid = np.asarray(np.load(f"{PDB_RAW}/subject_ids.npy", mmap_mode="r"))
    X_src   = np.load(f"{PDB_RAW}/X_ppg.npy",          mmap_mode="r")   # (N,1,128)
    A_src   = np.load(f"{PDB_RAW}/Y_abp.npy",          mmap_mode="r")   # (N,128)
    B_src   = np.load(f"{PDB_RAW}/Y_bp.npy",           mmap_mode="r")   # (N,2)
    D_src   = np.load(f"{PDB_RAW}/beat_durations.npy", mmap_mode="r")   # (N,)

    # ── plan the segments first (cheap, ints only) ──────────────────────────
    order = np.argsort(sid, kind="stable")
    s_sorted = sid[order]
    uniq, starts, counts = np.unique(s_sorted, return_index=True, return_counts=True)
    plan = []                                   # (src_beat_idx array, subj, seg_id)
    seg_id = 0
    for su, st, cn in zip(uniq, starts, counts):
        if cn < 2 * HALF + 1:                   # cannot form even one window
            continue
        L = min(SEG_LEN, cn)
        n_seg = min(SEG_PER_SUBJ, cn // L)
        # evenly spaced segment starts across the subject's recording
        if n_seg == 1:
            offs = [0]
        else:
            offs = np.linspace(0, cn - L, n_seg).astype(int)
        for o in offs:
            plan.append((order[st + o: st + o + L], int(su), seg_id))
            seg_id += 1
    n_sel = sum(len(p[0]) for p in plan)
    print(f"  planned {len(plan):,} segments / {len(set(p[1] for p in plan)):,} subjects "
          f"-> {n_sel:,} beats")

    # ── allocate on DISK, fill segment by segment ───────────────────────────
    om = np.lib.format.open_memmap
    Xo = om(f"{PDB_SUBSET}/X_ppg.npy", mode="w+", dtype=np.float32, shape=(n_sel, 1, 128))
    Ao = om(f"{PDB_SUBSET}/Y_abp.npy", mode="w+", dtype=np.float32, shape=(n_sel, 128))
    Bo = om(f"{PDB_SUBSET}/Y_bp.npy",  mode="w+", dtype=np.float32, shape=(n_sel, 2))
    Do = om(f"{PDB_SUBSET}/dur.npy",   mode="w+", dtype=np.float32, shape=(n_sel,))
    So = om(f"{PDB_SUBSET}/subj.npy",  mode="w+", dtype=np.int64,   shape=(n_sel,))
    Go = om(f"{PDB_SUBSET}/seg.npy",   mode="w+", dtype=np.int64,   shape=(n_sel,))

    w = 0
    for k, (idx, su, sg) in enumerate(plan):
        idx = np.sort(idx)                      # keep TEMPORAL order within a segment
        L = len(idx)
        Xo[w:w+L] = X_src[idx]
        Ao[w:w+L] = A_src[idx]
        Bo[w:w+L] = B_src[idx]
        Do[w:w+L] = D_src[idx]
        So[w:w+L] = su
        Go[w:w+L] = sg
        w += L
        if k % 2000 == 0:
            print(f"    {k:>6,}/{len(plan):,} segments  ({w:,} beats)", flush=True)
    for arr in (Xo, Ao, Bo, Do, So, Go):
        arr.flush()
    del Xo, Ao, Bo, Do, So, Go
    gc.collect()

    json.dump({**_WANT, "n_beats": int(n_sel), "n_segments": len(plan),
               "n_subjects": int(len(set(p[1] for p in plan)))},
              open(_MANIFEST, "w"), indent=1)
    sz = sum(os.path.getsize(os.path.join(PDB_SUBSET, f))
             for f in os.listdir(PDB_SUBSET) if f.endswith(".npy"))
    print(f"  ✅ wrote {n_sel:,} beats to {PDB_SUBSET}  ({sz/1e9:.2f} GB)  "
          f"in {time.time()-t0:.0f}s")

build_subset()

In [ ]:
# STAGE B — subject-level split on real subject IDs, plus window centres.
# This is the point of the notebook: PulseDB groups are patients, so a grouped
# split means what it says (on UCI the group was a record, and 90.9% of test
# records sat next to a train record). A window centre is valid iff its 11 beats
# lie in one segment.

_SPLIT_IDX = os.path.join(PDB_SUBSET, "split_idx.npz")

def build_split():
    subj = np.load(f"{PDB_SUBSET}/subj.npy")
    seg  = np.load(f"{PDB_SUBSET}/seg.npy")
    ybp  = np.load(f"{PDB_SUBSET}/Y_bp.npy")
    n    = len(subj)

    # valid window centres (vectorised)
    c = np.arange(HALF, n - HALF, STRIDE)
    c = c[seg[c - HALF] == seg[c + HALF]]
    print(f"  {len(c):,} valid window centres from {n:,} beats")

    # subject-level split, on the WINDOW CENTRES' subjects
    g = subj[c]
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SPLIT_SEED)
    tv, te = next(gss.split(c, groups=g))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC / (1 - TEST_FRAC),
                             random_state=SPLIT_SEED)
    _tr, _va = next(gss2.split(c[tv], groups=g[tv]))
    tr, va = tv[_tr], tv[_va]

    idx = dict(train=c[tr], val=c[va], test=c[te])
    subj_sets = {k: set(np.unique(subj[v]).tolist()) for k, v in idx.items()}
    assert not (subj_sets["train"] & subj_sets["test"]), "SUBJECT leakage train/test"
    assert not (subj_sets["train"] & subj_sets["val"]),  "SUBJECT leakage train/val"
    assert not (subj_sets["val"]   & subj_sets["test"]), "SUBJECT leakage val/test"
    print("  ✅ no subject appears in two splits — and here 'subject' really is a patient")

    for k, v in idx.items():
        s = ybp[v, 0]
        print(f"  {k:<6} {len(v):>8,} windows  {len(subj_sets[k]):>5,} subjects  "
              f"SBP {s.mean():6.1f} +/- {s.std():5.1f}")

    # stats from TRAIN beats only (no leakage), matching make_pre_split
    tr_beats = np.zeros(len(subj), bool)
    for cc in idx["train"]:
        tr_beats[cc - HALF: cc + HALF + 1] = True
    sbp_tr, dbp_tr = ybp[tr_beats, 0], ybp[tr_beats, 1]
    np.savez(_SPLIT_IDX, train=idx["train"], val=idx["val"], test=idx["test"],
             sbp_mu=sbp_tr.mean(), sbp_sig=sbp_tr.std(),
             dbp_mu=dbp_tr.mean(), dbp_sig=dbp_tr.std())
    print(f"  saved {_SPLIT_IDX}")
    print(f"  sbp_mu={sbp_tr.mean():.1f} sbp_sig={sbp_tr.std():.1f}  "
          f"dbp_mu={dbp_tr.mean():.1f} dbp_sig={dbp_tr.std():.1f}")

if not os.path.exists(_SPLIT_IDX):
    build_split()
else:
    print(f"  ✅ {_SPLIT_IDX} exists — skipping (delete it to rebuild)")

In [ ]:
# STAGE C — lazy streaming SPLITS.
# phase2 reaches the data through one function, _make_tensor_dataset. We build a
# Dataset that yields the same tuple but cuts windows on the fly from the memmap,
# and rebind that one name. Nothing else changes, so the model, losses and schedule
# stay bit-identical to the paper's.

class _LazyWindows:
    """Duck-types enough of an ndarray for phase2's `len(split.X)` / `.shape`."""
    def __init__(self, centres, half):
        self.centres, self.half = centres, half
    def __len__(self):
        return len(self.centres)
    @property
    def shape(self):
        return (len(self.centres), 2 * self.half + 1, 128)


def _q_proxy(ppg):
    """Rectified gradient, integral-normalised to sum~8 — identical to make_pre_split."""
    d = np.gradient(ppg, axis=-1)
    q = np.maximum(d, 0.0)
    a = np.trapezoid(q, axis=-1)[..., np.newaxis]
    a = np.where(a < 1e-6, 1e-6, a)
    return (q / a * 8.0).astype(np.float32)


class PulseDBWindows(Dataset):
    def __init__(self, centres, X_mm, A_mm, ybp, htn, subj, dur, base, half):
        self.c, self.X, self.A = centres, X_mm, A_mm
        self.ybp, self.htn, self.subj, self.dur, self.base = ybp, htn, subj, dur, base
        self.half = half
    def __len__(self):
        return len(self.c)
    def __getitem__(self, k):
        c  = int(self.c[k])
        sl = slice(c - self.half, c + self.half + 1)   # contiguous on disk
        x  = np.asarray(self.X[sl, 0, :], dtype=np.float32)   # (11,128)
        a  = np.asarray(self.A[sl],       dtype=np.float32)   # (11,128)
        q  = _q_proxy(x)
        return (torch.from_numpy(x), torch.from_numpy(q), torch.from_numpy(a),
                torch.from_numpy(self.ybp[k]), torch.tensor(self.htn[k]),
                torch.tensor(self.subj[k], dtype=torch.long),
                torch.tensor(self.dur[k]), torch.tensor(0, dtype=torch.long),
                torch.tensor(self.base[k]))


def make_pulsedb_splits():
    idx  = np.load(_SPLIT_IDX)
    subj = np.load(f"{PDB_SUBSET}/subj.npy")
    ybp  = np.load(f"{PDB_SUBSET}/Y_bp.npy")
    dur  = np.load(f"{PDB_SUBSET}/dur.npy")
    A_mm = np.load(f"{PDB_SUBSET}/Y_abp.npy", mmap_mode="r")
    X_mm = np.load(f"{PDB_SUBSET}/X_ppg.npy", mmap_mode="r")

    # subject re-index 0..n-1 (the embedding is indexed by this)
    uniq  = np.unique(subj)
    remap = {s: i for i, s in enumerate(uniq)}
    subj_ix = np.array([remap[s] for s in subj], dtype=np.int64)

    # per-subject baseline: MAP of that subject's FIRST beat, pop-norm space
    # (same definition as make_pre_split -- do not "improve" it, it must match)
    first = {}
    for s in uniq:
        first[s] = int(np.where(subj == s)[0][0])
    base_by_subj = {s: float((A_mm[i].max() + A_mm[i].min()) / 2.0) for s, i in first.items()}
    base_all = np.array([base_by_subj[s] for s in subj], dtype=np.float32)

    htn_all = ((ybp[:, 0] >= HTN_SBP) | (ybp[:, 1] >= HTN_DBP)).astype(np.float32)

    def _mk(centres):
        centres = np.asarray(centres)
        sp = Split.__new__(Split)
        sp.X    = _LazyWindows(centres, HALF)      # lazy: windows cut on the fly
        sp.Q    = _LazyWindows(centres, HALF)
        sp.Yabp = _LazyWindows(centres, HALF)
        sp.Ybp  = ybp[centres]                     # eager, small
        sp.dur  = dur[centres]
        sp.htn  = htn_all[centres]
        sp.subj = subj_ix[centres]
        sp.dom  = np.zeros(len(centres), dtype=np.int64)
        sp.base = base_all[centres]
        sp._ds  = PulseDBWindows(centres, X_mm, A_mm, ybp[centres], htn_all[centres],
                                 subj_ix[centres], dur[centres], base_all[centres], HALF)
        return sp

    sp = DataSplits(
        train=_mk(idx["train"]), val=_mk(idx["val"]), test=_mk(idx["test"]),
        pop_mu=66.2, pop_sig=23.6,                 # phase1 population constants
        sbp_mu=float(idx["sbp_mu"]), sbp_sig=float(idx["sbp_sig"]),
        dbp_mu=float(idx["dbp_mu"]), dbp_sig=float(idx["dbp_sig"]),
        n_subjects=int(len(uniq)),
    )
    return sp


# ── the one-line swap that makes all of phase2 work on PulseDB ───────────────
_orig_make_tensor_dataset = _make_tensor_dataset

def _make_tensor_dataset(split):                       # noqa: F811  (deliberate override)
    ds = getattr(split, "_ds", None)
    return ds if ds is not None else _orig_make_tensor_dataset(split)

SPLITS = make_pulsedb_splits()
print(f"  windows — train:{len(SPLITS.train.X):,}  val:{len(SPLITS.val.X):,}  "
      f"test:{len(SPLITS.test.X):,}")
print(f"  n_subjects={SPLITS.n_subjects:,}   sbp_mu={SPLITS.sbp_mu:.1f}  "
      f"sbp_sig={SPLITS.sbp_sig:.1f}")

# smoke-test one batch through the real model before committing to a long run
_dl = DataLoader(_make_tensor_dataset(SPLITS.val), batch_size=8, shuffle=False)
_b = next(iter(_dl))
print(f"  batch tuple: {[tuple(t.shape) for t in _b]}")
assert len(_b) == 9 and _b[0].shape[1:] == (2 * HALF + 1, 128), "batch shape mismatch"
print("  ✅ streaming loader yields the exact 9-tuple phase2's _run_epoch expects")
del _dl, _b; gc.collect()

In [ ]:
# STAGE D — train, using phase2's own `_two_phase_train` (not a reimplementation).
# Idempotent: the best checkpoint is written to CKPT_DIR, so re-running resumes.

NUM_WORKERS = 0
TAG         = "pulsedb_v192"

print(f"  DEVICE={DEVICE}  batch={ARCH.batch}  workers={NUM_WORKERS}")
print(f"  epochs: {TRAIN.epochs_p1} + {TRAIN.epochs_p2} = {TRAIN.epochs_p1+TRAIN.epochs_p2}")
print(f"  steps/epoch ~ {len(SPLITS.train.X)//ARCH.batch:,}")
print(f"  checkpoints -> {CKPT_DIR}\n")

mdl = DiBPNet().to(DEVICE)
mdl._set_stats_from_splits(SPLITS)
print(f"  model parameters: {mdl.n_params():,}\n")

if _ckpt_exists(f"{TAG}_best"):
    print(f"  ✅ {TAG}_best.pt exists — loading it, skipping training "
          f"(delete the checkpoint to force a retrain)")
    mdl, hist = load_checkpoint(f"{TAG}_best")
    if not hist:
        hist = _empty_hist()
else:
    hist = _empty_hist()
    t0 = time.time()
    mdl, hist = _two_phase_train(mdl, TAG, hist, print_prefix="")
    print(f"\n  ✅ training finished in {(time.time()-t0)/60:.1f} min")
    save_hist_csv(hist, tag=TAG)

In [ ]:
# STAGE E — the number this notebook exists to produce.
# A PulseDB in-domain MAE much worse than UCI's 4.18 is the measurement, not a
# failure: the split is genuinely subject-independent here (it was not on UCI), and
# PulseDB's BP moves within a session (SBP ICC 0.425) while a UCI "subject" is a
# short segment whose BP barely moves (ICC 0.955). Do not tune until the two agree.

sbp_v, dbp_v, auc_v = quick_metrics(mdl, SPLITS.val)
sbp_t, dbp_t, auc_t = quick_metrics(mdl, SPLITS.test)

print("=" * 78)
print("  DiBP-Net trained on PulseDB — SUBJECT-independent (real patient IDs)")
print("=" * 78)
print(f"  {'split':<8} {'SBP MAE':>9} {'DBP MAE':>9} {'HTN AUC':>9}")
print("  " + "-" * 40)
print(f"  {'val':<8} {sbp_v:>9.2f} {dbp_v:>9.2f} {auc_v:>9.4f}")
print(f"  {'test':<8} {sbp_t:>9.2f} {dbp_t:>9.2f} {auc_t:>9.4f}")

print("\n  ── against the UCI numbers currently in the paper ──")
print(f"  {'dataset':<22} {'split unit':<12} {'SBP':>7} {'DBP':>7} {'AUC':>8}")
print("  " + "-" * 60)
print(f"  {'UCI (paper)':<22} {'record':<12} {4.18:>7.2f} {2.55:>7.2f} {0.9608:>8.4f}")
print(f"  {'PulseDB (this run)':<22} {'PATIENT':<12} {sbp_t:>7.2f} {dbp_t:>7.2f} {auc_t:>8.4f}")
print(f"  {'delta':<22} {'':<12} {sbp_t-4.18:>+7.2f} {dbp_t-2.55:>+7.2f} {auc_t-0.9608:>+8.4f}")
print("\n  The UCI row is NOT subject-independent: 90.9% of its test records have a")
print("  train record immediately adjacent, and adjacent records are the same patient")
print("  (SBP r = +0.709). The two rows measure different things. Say so in the paper;")
print("  do not present the delta as a domain gap.")

pd.DataFrame([
    dict(dataset="UCI",     split_unit="record",  sbp=4.18,  dbp=2.55,  auc=0.9608, source="paper"),
    dict(dataset="PulseDB", split_unit="patient", sbp=sbp_t, dbp=dbp_t, auc=auc_t,  source=TAG),
]).to_csv(os.path.join(EXPORT_DIR, "pulsedb_vs_uci.csv"), index=False)
print(f"\n  ✅ saved -> {os.path.join(EXPORT_DIR, 'pulsedb_vs_uci.csv')}")

In [ ]:
# STAGE F — split-effect control: retrain on the SAME PulseDB subset but with a
# record-level (segment-grouped) split instead of the subject-level one.
# Stage E differs from UCI on two axes at once (dataset AND split); this adds the
# middle point so they separate:
#     Split Effect   = PulseDB(subject) - PulseDB(record)   [same data; causal]
#     Dataset Effect = PulseDB(record)  - UCI(record)        [both leaky; suggestive]
# For AUC this is decisive: if the collapse were identity-memorisation, a
# record-level split should push AUC back toward UCI's 0.96; if PulseDB HTN is
# genuinely harder, it stays ~0.78. Protocol is identical to Stage D; only the
# grouping changes (seg[] instead of subj[]).

RUN_RECORD_CONTROL = True       # flip to True to launch the ~10 h record-level control

_SPLIT_IDX_RECORD = os.path.join(PDB_SUBSET, "split_idx_record.npz")
TAG_RECORD        = "pulsedb_v192_record"


def build_split_record():
    """UCI-analogue split: group WINDOW CENTRES by segment, not by subject.

    Identical to build_split() except for the grouping key, so the only thing
    that can move the numbers is the split itself.
    """
    subj = np.load(f"{PDB_SUBSET}/subj.npy")
    seg  = np.load(f"{PDB_SUBSET}/seg.npy")
    ybp  = np.load(f"{PDB_SUBSET}/Y_bp.npy")
    n    = len(subj)

    # valid window centres — the SAME guard and stride as the subject-level split
    c = np.arange(HALF, n - HALF, STRIDE)
    c = c[seg[c - HALF] == seg[c + HALF]]
    print(f"  {len(c):,} valid window centres from {n:,} beats")

    # group by SEGMENT (the record analogue). A subject owns many segments, so its
    # segments scatter across train/val/test — the same patient leaks across the
    # split, exactly as adjacent UCI records (same patient) do.
    g = seg[c]
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SPLIT_SEED)
    tv, te = next(gss.split(c, groups=g))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC / (1 - TEST_FRAC),
                             random_state=SPLIT_SEED)
    _tr, _va = next(gss2.split(c[tv], groups=g[tv]))
    tr, va = tv[_tr], tv[_va]

    idx = dict(train=c[tr], val=c[va], test=c[te])

    # segments must NOT cross splits (a window never spans two segments, so this
    # is the integrity guard); subjects SHOULD cross — that overlap is the leak we
    # are deliberately introducing and want to quantify.
    seg_sets  = {k: set(np.unique(seg[v]).tolist())  for k, v in idx.items()}
    subj_sets = {k: set(np.unique(subj[v]).tolist()) for k, v in idx.items()}
    assert not (seg_sets["train"] & seg_sets["test"]), "SEGMENT crossed the split"
    assert not (seg_sets["train"] & seg_sets["val"]),  "SEGMENT crossed the split"
    overlap = subj_sets["train"] & subj_sets["test"]
    frac = 100.0 * len(overlap) / max(1, len(subj_sets["test"]))
    print(f"  record-level (segment-grouped) split — this is the UCI analogue")
    print(f"  {len(overlap):,}/{len(subj_sets['test']):,} test patients also appear in "
          f"train ({frac:.1f}% leak) — intended, this is the knob under test")

    for k, v in idx.items():
        s = ybp[v, 0]
        print(f"  {k:<6} {len(v):>8,} windows  {len(seg_sets[k]):>5,} segments  "
              f"{len(subj_sets[k]):>5,} subjects  SBP {s.mean():6.1f} +/- {s.std():5.1f}")

    # train-only stats, same as build_split (no leakage into normalisation)
    tr_beats = np.zeros(len(subj), bool)
    for cc in idx["train"]:
        tr_beats[cc - HALF: cc + HALF + 1] = True
    sbp_tr, dbp_tr = ybp[tr_beats, 0], ybp[tr_beats, 1]
    np.savez(_SPLIT_IDX_RECORD, train=idx["train"], val=idx["val"], test=idx["test"],
             sbp_mu=sbp_tr.mean(), sbp_sig=sbp_tr.std(),
             dbp_mu=dbp_tr.mean(), dbp_sig=dbp_tr.std())
    print(f"  saved {_SPLIT_IDX_RECORD}")


if not RUN_RECORD_CONTROL:
    print("  RUN_RECORD_CONTROL is False — record-level control not launched.")
    print("  Review the code above, then set RUN_RECORD_CONTROL = True and re-run")
    print("  this cell to start the ~10 h run (isolated from the Stage D outputs).")
else:
    # ── 1. build the record-level split (idempotent) ─────────────────────────
    if not os.path.exists(_SPLIT_IDX_RECORD):
        build_split_record()
    else:
        print(f"  ✅ {_SPLIT_IDX_RECORD} exists — skipping (delete to rebuild)")

    # ── 2. swap the global SPLITS to record-level, reusing cell 5's builder ──
    #    make_pulsedb_splits() reads the module global _SPLIT_IDX; point it at the
    #    record file for one call, then restore the constant immediately. The
    #    trainer and quick_metrics read the global SPLITS, so that is what must
    #    change for the run.
    SPLITS_SUBJECT   = SPLITS                 # keep the Stage C subject-level splits
    _SPLIT_IDX_SAVED = _SPLIT_IDX
    _SPLIT_IDX = _SPLIT_IDX_RECORD
    SPLITS = make_pulsedb_splits()
    _SPLIT_IDX = _SPLIT_IDX_SAVED
    print(f"  record-level SPLITS — train:{len(SPLITS.train.X):,}  "
          f"val:{len(SPLITS.val.X):,}  test:{len(SPLITS.test.X):,}")

    # ── 3. train, bit-identical protocol, isolated TAG ───────────────────────
    print(f"\n  DEVICE={DEVICE}  batch={ARCH.batch}  TAG={TAG_RECORD}")
    print(f"  epochs: {TRAIN.epochs_p1} + {TRAIN.epochs_p2}   "
          f"steps/epoch ~ {len(SPLITS.train.X)//ARCH.batch:,}")
    print(f"  checkpoints -> {CKPT_DIR}\n")
    mdl_rec = DiBPNet().to(DEVICE)
    mdl_rec._set_stats_from_splits(SPLITS)
    if _ckpt_exists(f"{TAG_RECORD}_best"):
        print(f"  ✅ {TAG_RECORD}_best.pt exists — loading it, skipping training "
              f"(delete the checkpoint to force a retrain)")
        mdl_rec, hist_rec = load_checkpoint(f"{TAG_RECORD}_best")
        if not hist_rec:
            hist_rec = _empty_hist()
    else:
        hist_rec = _empty_hist()
        _t0 = time.time()
        mdl_rec, hist_rec = _two_phase_train(mdl_rec, TAG_RECORD, hist_rec, print_prefix="[rec] ")
        print(f"\n  ✅ record-level training finished in {(time.time()-_t0)/60:.1f} min")
        save_hist_csv(hist_rec, tag=TAG_RECORD)

    # ── 4. evaluate the record-level model (same selection rule as Stage E: the
    #       model _two_phase_train returns is the phase-2 best, so both runs are
    #       compared on the same footing) ────────────────────────────────────
    sbp_rv, dbp_rv, auc_rv = quick_metrics(mdl_rec, SPLITS.val)
    sbp_rt, dbp_rt, auc_rt = quick_metrics(mdl_rec, SPLITS.test)

    # subject-level numbers: read back from Stage E's CSV so nothing goes stale
    _subj = pd.read_csv(os.path.join(EXPORT_DIR, "pulsedb_vs_uci.csv"))
    _srow = _subj[_subj.dataset == "PulseDB"].iloc[0]
    sbp_sj, dbp_sj, auc_sj = float(_srow.sbp), float(_srow.dbp), float(_srow.auc)
    UCI_SBP, UCI_DBP, UCI_AUC = 4.18, 2.55, 0.9608

    print("\n" + "=" * 78)
    print("  SPLIT-EFFECT DECOMPOSITION  (SBP MAE / DBP MAE / HTN AUC, test)")
    print("=" * 78)
    print(f"  {'dataset':<10} {'split':<9} {'SBP':>7} {'DBP':>7} {'AUC':>8}   note")
    print("  " + "-" * 68)
    print(f"  {'UCI':<10} {'record':<9} {UCI_SBP:>7.2f} {UCI_DBP:>7.2f} {UCI_AUC:>8.4f}   paper (leaky)")
    print(f"  {'PulseDB':<10} {'record':<9} {sbp_rt:>7.2f} {dbp_rt:>7.2f} {auc_rt:>8.4f}   this cell (leaky)")
    print(f"  {'PulseDB':<10} {'subject':<9} {sbp_sj:>7.2f} {dbp_sj:>7.2f} {auc_sj:>8.4f}   Stage E (clean)")
    print("  " + "-" * 68)
    print(f"  Split Effect  (PulseDB subject − record, SAME data, CAUSAL):")
    print(f"      SBP {sbp_sj - sbp_rt:>+6.2f}   DBP {dbp_sj - dbp_rt:>+6.2f}   "
          f"AUC {auc_sj - auc_rt:>+7.4f}")
    print(f"  Dataset Effect(PulseDB record − UCI record, CONFOUNDED, suggestive):")
    print(f"      SBP {sbp_rt - UCI_SBP:>+6.2f}   DBP {dbp_rt - UCI_DBP:>+6.2f}   "
          f"AUC {auc_rt - UCI_AUC:>+7.4f}")
    print("\n  Reading: the Split-Effect row is the only clean causal contrast (dataset,")
    print("  preprocessing and cohort are all held fixed). The Dataset-Effect row still")
    print("  confounds corpus identity with the record/patient question, so report it as")
    print("  context, never as a single-variable measurement. Read MAE and AUC")
    print("  separately: MAE may be task-difficulty-dominated while AUC is leak-dominated.")

    pd.DataFrame([
        dict(dataset="UCI",     split="record",  sbp=UCI_SBP, dbp=UCI_DBP, auc=UCI_AUC, source="paper"),
        dict(dataset="PulseDB", split="record",  sbp=sbp_rt,  dbp=dbp_rt,  auc=auc_rt,  source=TAG_RECORD),
        dict(dataset="PulseDB", split="subject", sbp=sbp_sj,  dbp=dbp_sj,  auc=auc_sj,  source=TAG),
    ]).to_csv(os.path.join(EXPORT_DIR, "pulsedb_split_decomposition.csv"), index=False)
    print(f"\n  ✅ saved -> {os.path.join(EXPORT_DIR, 'pulsedb_split_decomposition.csv')}")

    # ── 5. restore the notebook's default in-memory state ────────────────────
    SPLITS = SPLITS_SUBJECT
    print("  restored subject-level SPLITS as the in-memory default.")

In [ ]:
# STAGE F-eval — record vs subject at all three calibration stages, from disk.
# The real pipeline chains SCal on top of GCal (Post = Raw -> GCal -> SCal), so this
# reproduces that with phase2's own calibration functions, fitting GCal on each
# split's own val. UCI comes from the v192 export.

from sklearn.linear_model import Ridge                    # de-risk cell-25 exec
from sklearn.isotonic import IsotonicRegression
from typing import Optional
import numpy as np

# 1. bring in the REAL calibration definitions (cells 25 + 27 are pure defs).
for _ci in (21, 25, 27):
    exec(compile("".join(json.load(open(PHASE2_NB))["cells"][_ci]["source"]),
                 f"<{PHASE2_NB} cell {_ci}>", "exec"), globals())
for _need in ("CALIB_STATE", "fit_ridge_on_val", "fit_isotonic_on_val",
              "get_raw_predictions", "apply_subject_calibration",
              "_build_subj_cache_for_split"):
    assert _need in globals(), f"calibration def missing after exec: {_need}"


def _materialize_sorted(sp):
    """Turn a lazy PulseDB Split into an eager one, ordered by (subject, centre)."""
    centres = np.asarray(sp._ds.c)
    order   = np.lexsort((centres, np.asarray(sp.subj)))   # temporal within subject
    centres = centres[order]
    for f in ("Ybp", "dur", "htn", "subj", "dom", "base"):
        setattr(sp, f, np.asarray(getattr(sp, f))[order])
    X_mm = np.load(f"{PDB_SUBSET}/X_ppg.npy", mmap_mode="r")
    A_mm = np.load(f"{PDB_SUBSET}/Y_abp.npy", mmap_mode="r")
    N, W = len(centres), 2 * HALF + 1
    X = np.empty((N, W, 128), np.float32); Q = np.empty((N, W, 128), np.float32)
    Yabp = np.empty((N, W, 128), np.float32)
    for k, cc in enumerate(centres):
        cc = int(cc); s = slice(cc - HALF, cc + HALF + 1)
        x = np.asarray(X_mm[s, 0, :], np.float32)
        X[k] = x; Q[k] = _q_proxy(x); Yabp[k] = np.asarray(A_mm[s], np.float32)
    sp.X, sp.Q, sp.Yabp, sp._ds = X, Q, Yabp, None          # eager now
    return sp


def _auc_on_test(mdl, split):
    mdl.eval(); S, H = [], []
    dl = DataLoader(_make_tensor_dataset(split), batch_size=ARCH.batch, shuffle=False)
    with torch.no_grad():
        for batch in dl:
            bx, bq, _, _, bht, bsubj, _, _, bbase = _to_device(batch)
            q_c = bq[:, ARCH.n_beats // 2]
            if q_c.dim() == 3: q_c = q_c.squeeze(1)
            out = mdl(bx, bsubj, bbase, q_c, 0.0)
            S.append(_blend_htn_scores(out[6], out[13]).cpu().numpy()); H.append(bht.cpu().numpy())
    S, H = np.concatenate(S), np.concatenate(H)
    return float(roc_auc_score(H, S)) if len(np.unique(H)) > 1 else float("nan")


def eval_from_ckpt(split_idx_path, ckpt_tag):
    """Raw / GCal / SCal MAE for a checkpoint on the split it was trained on."""
    global _SPLIT_IDX, SPLITS
    _saved = _SPLIT_IDX
    _SPLIT_IDX = split_idx_path
    SP = make_pulsedb_splits()
    _SPLIT_IDX = _saved
    _materialize_sorted(SP.val); _materialize_sorted(SP.test)
    SPLITS = SP                                     # load_checkpoint + calib read this

    mdl_c, _ = load_checkpoint(ckpt_tag)
    if mdl_c is None:
        print(f"  (missing checkpoint: {ckpt_tag})"); return None

    # GCal: fit Ridge+Iso on THIS split's val (leave CALIB_STATE populated)
    CALIB_STATE.ridge_sbp = CALIB_STATE.ridge_dbp = None
    CALIB_STATE.iso_s = CALIB_STATE.iso_d = None
    _build_subj_cache_for_split(mdl_c, SPLITS.val)
    fit_ridge_on_val(mdl_c)
    pv, _ = get_raw_predictions(mdl_c, SPLITS.val)
    fit_isotonic_on_val(pv)

    # test predictions at each stage (personalise from test ref beats first)
    _build_subj_cache_for_split(mdl_c, SPLITS.test)
    gcal, raw = get_raw_predictions(mdl_c, SPLITS.test)      # gcal=Ridge+Iso, raw=bp_ens
    Ybp = SPLITS.test.Ybp
    scal, mask = apply_subject_calibration(gcal, Ybp, SPLITS.test.subj)  # SCal on GCal

    mae = lambda P, m=slice(None): (float(np.abs(P[m, 0] - Ybp[m, 0]).mean()),
                                    float(np.abs(P[m, 1] - Ybp[m, 1]).mean()))
    raw_sbp,  raw_dbp  = mae(raw)
    gcal_sbp, gcal_dbp = mae(gcal)
    scal_sbp, scal_dbp = mae(scal, mask)
    return dict(tag=ckpt_tag, n=len(raw), n_eval=int(mask.sum()),
                raw_sbp=raw_sbp, raw_dbp=raw_dbp, gcal_sbp=gcal_sbp, gcal_dbp=gcal_dbp,
                scal_sbp=scal_sbp, scal_dbp=scal_dbp, auc=_auc_on_test(mdl_c, SPLITS.test))


_SPLITS_BEFORE = SPLITS
if not os.path.exists(_SPLIT_IDX_RECORD):
    build_split_record()

subj_r = eval_from_ckpt(_SPLIT_IDX,        f"{TAG}_best")          # subject split, Stage-D ckpt
rec_r  = eval_from_ckpt(_SPLIT_IDX_RECORD, f"{TAG_RECORD}_best")   # record  split, Stage-F ckpt

# UCI DiBP-Net Raw / Pre(=GCal) / Post(=SCal) straight from the real export
_uci = pd.read_csv("outputs/phase_2_v192/exports/v192/overall_summary.csv")
_ur  = _uci[(_uci["Model"].str.contains("v192")) & (_uci["Split"] == "Test")].iloc[0]
uci = dict(raw_sbp=float(_ur["Beat SBP (Raw)"]),  gcal_sbp=float(_ur["Beat SBP (Pre)"]),
           scal_sbp=float(_ur["Beat SBP (Post)"]), auc=float(_ur["AUC (HTN)"]))

def _row(name, split, d):
    print(f"  {name:<8} {split:<8} {d['raw_sbp']:>7.2f} {d['gcal_sbp']:>7.2f} {d['scal_sbp']:>8.2f}   {d['auc']:>7.4f}")

print("=" * 78)
print("  RECORD vs SUBJECT — Raw / GCal / SCal (SBP MAE, test)")
print("=" * 78)
print(f"  {'dataset':<8} {'split':<8} {'Raw':>7} {'GCal':>7} {'SCal':>8}   {'AUC':>7}")
print("  " + "-" * 60)
_row("UCI",     "record",  uci)
_row("PulseDB", "record",  rec_r)
_row("PulseDB", "subject", subj_r)
print("  " + "-" * 60)
print("  Split Effect (subject - record, SAME PulseDB data, CAUSAL):")
print(f"      Raw {subj_r['raw_sbp']-rec_r['raw_sbp']:+.2f}   GCal {subj_r['gcal_sbp']-rec_r['gcal_sbp']:+.2f}"
      f"   SCal {subj_r['scal_sbp']-rec_r['scal_sbp']:+.2f}   AUC {subj_r['auc']-rec_r['auc']:+.4f}")
print("  Dataset Effect (PulseDB record - UCI record, CONFOUNDED, suggestive):")
print(f"      Raw {rec_r['raw_sbp']-uci['raw_sbp']:+.2f}   GCal {rec_r['gcal_sbp']-uci['gcal_sbp']:+.2f}"
      f"   SCal {rec_r['scal_sbp']-uci['scal_sbp']:+.2f}")
print(f"\n  Post/SCal is SCal-on-GCal (the paper's chain). SCal scored on the eval beats "
      f"({rec_r['n_eval']:,}/{rec_r['n']:,} record, {subj_r['n_eval']:,}/{subj_r['n']:,} subject); "
      f"Raw/GCal on all. UCI from v192 overall_summary.csv.")

pd.DataFrame([
    dict(dataset="UCI",     split="record",  raw_sbp=uci["raw_sbp"],    gcal_sbp=uci["gcal_sbp"],    scal_sbp=uci["scal_sbp"],    auc=uci["auc"],    source="v192_export"),
    dict(dataset="PulseDB", split="record",  raw_sbp=rec_r["raw_sbp"],  gcal_sbp=rec_r["gcal_sbp"],  scal_sbp=rec_r["scal_sbp"],  auc=rec_r["auc"],  source=rec_r["tag"]),
    dict(dataset="PulseDB", split="subject", raw_sbp=subj_r["raw_sbp"], gcal_sbp=subj_r["gcal_sbp"], scal_sbp=subj_r["scal_sbp"], auc=subj_r["auc"], source=subj_r["tag"]),
]).to_csv(os.path.join(EXPORT_DIR, "pulsedb_split_decomposition.csv"), index=False)
print(f"\n  ✅ saved -> {os.path.join(EXPORT_DIR, 'pulsedb_split_decomposition.csv')}")

SPLITS = _SPLITS_BEFORE
print("  restored the previous in-memory SPLITS.")